generating data for the project. test first

In [2]:
"""
test_generator.py

TEST SCRIPT - Does NOT save any files.
Generates sample data and prints the first few rows to the console.
"""

import json
import random
import uuid
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('sv_SE')

# ===================================================================
# CONFIGURATION - TEST MODE
# ===================================================================

NUM_TRUCKS_PER_OEM = 20
NUM_OEMS = 5
NUM_AUTONOMOUS = 20

# Small test counts
TEST_ROWS_TRADITIONAL = 100
TEST_ROWS_AUTONOMOUS = 100

# ===================================================================
# OEM DISTRIBUTION (Real field mappings from research)
# ===================================================================

OEMS = {
    "Tesla": {
        "models": ["Semi Electric"],
        "battery_options": [500, 600, 750],
        "telemetry_fields": {
            "energy": "battery_energy_used",
            "energy_type": "cumulative",
            "soc": "soc_percentage",
            "speed": "vehicle_speed",
            "location": "gps_coordinates"
        }
    },
    "BYD": {
        "models": ["8TT", "8R"],
        "battery_options": [300, 350, 400],
        "telemetry_fields": {
            "energy": "energy_consumption",
            "energy_type": "cumulative",
            "soc": "battery_soc",
            "speed": "speed_kmh",
            "location": "location"
        }
    },
    "Scania": {
        "models": ["BEV 40R", "BEV 45R"],
        "battery_options": [300, 450, 600],
        "telemetry_fields": {
            "energy": "Total Discharged Energy",
            "energy_type": "cumulative",
            "soc": "EV Battery State Of Charge",
            "speed": "Vehicle Speed",
            "location": "coordinates"
        }
    },
    "Mercedes": {
        "models": ["eActros 300", "eActros 400"],
        "battery_options": [300, 400],
        "telemetry_fields": {
            "energy": "energy_consumed",
            "energy_type": "cumulative",
            "soc": "soc",
            "speed": "speed",
            "location": "gps"
        }
    },
    "DAF": {
        "models": ["CF Electric", "XF Electric"],
        "battery_options": [300, 450],
        "telemetry_fields": {
            "energy": "Energy consumed",
            "energy_type": "fms_standard",
            "soc": "State of charge",
            "speed": "Vehicle speed",
            "location": "position"
        }
    }
}

# ===================================================================
# GENERATE TRUCKS (Traditional)
# ===================================================================

def generate_traditional_trucks():
    """Generates 100 trucks across 5 OEMs."""
    trucks = []
    for oem_name, oem_data in OEMS.items():
        for i in range(NUM_TRUCKS_PER_OEM):
            purchase_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 1000))
            age_days = (datetime(2026, 8, 25) - purchase_date).days
            soh = max(85, min(100, 100 - age_days * 0.005))
            
            trucks.append({
                "truck_id": f"EIN-TRK-{oem_name[:3].upper()}-{i+1:03d}",
                "vin": fake.vin(),
                "oem": oem_name,
                "model": random.choice(oem_data["models"]),
                "battery_capacity_kwh": random.choice(oem_data["battery_options"]),
                "purchase_date": purchase_date.isoformat(),
                "home_depot_id": random.choice(["STO", "GOT", "MAL", "JKP", "HEL", "ORE"]),
                "vehicle_type": "human_driven",
                "initial_state_of_health_pct": round(soh, 1)
            })
    return trucks

# ===================================================================
# GENERATE AUTONOMOUS PODS
# ===================================================================

def generate_autonomous_pods():
    """Generates 20 autonomous pods."""
    pods = []
    for i in range(NUM_AUTONOMOUS):
        purchase_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 600))
        age_days = (datetime(2026, 8, 25) - purchase_date).days
        soh = max(92, min(100, 100 - age_days * 0.003))
        
        pods.append({
            "pod_id": f"EIN-POD-{i+1:03d}",
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "battery_capacity_kwh": random.choice([250, 300, 320]),
            "sensor_suite": ["lidar", "radar", "cameras", "gps", "inertial"],
            "autonomy_level": "Level 4",
            "max_range_km": random.choice([500, 600, 650]),
            "deployment_date": purchase_date.isoformat(),
            "home_depot_id": random.choice(["STO", "GOT", "MAL", "JKP"]),
            "initial_state_of_health_pct": round(soh, 1)
        })
    return pods

# ===================================================================
# GENERATE TRADITIONAL TELEMETRY (Test Sample)
# ===================================================================

def generate_traditional_telemetry_sample(trucks, num_rows=TEST_ROWS_TRADITIONAL):
    """Generates sample traditional telemetry."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        truck = random.choice(trucks)
        oem = truck["oem"]
        oem_fields = OEMS[oem]["telemetry_fields"]
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "truck_id": truck["truck_id"],
            "oem": oem,
            "vehicle_type": "human_driven",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            oem_fields["soc"]: round(random.uniform(20, 95), 1),
            oem_fields["speed"]: round(random.uniform(30, 85), 1),
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "ambient_temp_c": round(random.uniform(-5, 20), 1),
            "cargo_weight_kg": random.randint(5000, 30000),
            "is_charging": random.random() < 0.1,
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # OEM-specific energy field
        payload[oem_fields["energy"]] = round(random.uniform(0, 5000), 2)

        # Planted defects
        if random.random() < 0.0005:
            payload[oem_fields["soc"]] = random.choice([0, 100])
        if random.random() < 0.008:
            payload["cargo_weight_kg"] = None
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload[oem_fields["soc"]] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

    return pings

# ===================================================================
# GENERATE AUTONOMOUS TELEMETRY (Test Sample)
# ===================================================================

def generate_autonomous_telemetry_sample(pods, num_rows=TEST_ROWS_AUTONOMOUS):
    """Generates sample autonomous pod telemetry."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        pod = random.choice(pods)
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "pod_id": pod["pod_id"],
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "speed_kmh": round(random.uniform(30, 85), 1),
            "state_of_charge_pct": round(random.uniform(20, 95), 1),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "is_charging": random.random() < 0.1,
            "cargo_weight_kg": random.randint(5000, 30000),
            
            # Autonomous-specific fields
            "perception": {
                "objects_detected": random.randint(0, 25),
                "closest_object_distance_m": round(random.uniform(5, 150), 1)
            },
            "path_planning": {
                "planned_steering_angle_deg": round(random.uniform(-10, 10), 1),
                "planned_acceleration_ms2": round(random.uniform(-2, 2), 1)
            },
            "safety": {
                "system_health": "normal",
                "fallback_mode_active": random.random() < 0.01,
                "emergency_stop_triggered": False
            },
            "mission_status": random.choice(["en_route", "charging", "awaiting_remote", "maintenance"]),
            "remote_monitor_id": f"RM-{random.randint(1, 50):04d}",
            "safety_confidence_score": round(random.uniform(0.85, 0.99), 2),
            
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # Planted defects
        if random.random() < 0.0005:
            payload["state_of_charge_pct"] = random.choice([0, 100])
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload["state_of_charge_pct"] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

    return pings

# ===================================================================
# MAIN TEST
# ===================================================================

def main():
    print("\n" + "="*70)
    print("🧪 TESTING GENERATOR - No Files Will Be Saved")
    print("="*70 + "\n")
    
    # 1. Generate trucks
    print("📦 Generating 100 traditional trucks (20 per OEM)...")
    trucks = generate_traditional_trucks()
    print(f"   ✅ Generated {len(trucks)} traditional trucks")
    
    # 2. Generate pods
    print("📦 Generating 20 autonomous pods...")
    pods = generate_autonomous_pods()
    print(f"   ✅ Generated {len(pods)} autonomous pods")
    
    # 3. Generate sample telemetry
    print(f"\n📡 Generating {TEST_ROWS_TRADITIONAL} traditional telemetry rows...")
    traditional_pings = generate_traditional_telemetry_sample(trucks, TEST_ROWS_TRADITIONAL)
    print(f"   ✅ Generated {len(traditional_pings)} rows")
    
    print(f"\n📡 Generating {TEST_ROWS_AUTONOMOUS} autonomous telemetry rows...")
    autonomous_pings = generate_autonomous_telemetry_sample(pods, TEST_ROWS_AUTONOMOUS)
    print(f"   ✅ Generated {len(autonomous_pings)} rows")
    
    # 4. Print samples
    print("\n" + "-"*70)
    print("📋 SAMPLE TRADITIONAL TELEMETRY (First 2 rows):")
    print("-"*70)
    for i, ping in enumerate(traditional_pings[:2]):
        print(f"\nRow {i+1} (OEM: {ping.get('oem', 'Unknown')}):")
        print(f"  event_id: {ping.get('event_id')}")
        print(f"  truck_id: {ping.get('truck_id')}")
        print(f"  timestamp: {ping.get('timestamp')}")
        print(f"  latitude: {ping.get('latitude')}")
        print(f"  longitude: {ping.get('longitude')}")
        print(f"  alerts: {ping.get('alerts')}")
    
    print("\n" + "-"*70)
    print("📋 SAMPLE AUTONOMOUS TELEMETRY (First 2 rows):")
    print("-"*70)
    for i, ping in enumerate(autonomous_pings[:2]):
        print(f"\nRow {i+1} (Pod: {ping.get('pod_id')}):")
        print(f"  event_id: {ping.get('event_id')}")
        print(f"  pod_id: {ping.get('pod_id')}")
        print(f"  timestamp: {ping.get('timestamp')}")
        print(f"  latitude: {ping.get('latitude')}")
        print(f"  longitude: {ping.get('longitude')}")
        print(f"  mission_status: {ping.get('mission_status')}")
        print(f"  safety_confidence_score: {ping.get('safety_confidence_score')}")
        print(f"  perception_objects: {ping.get('perception', {}).get('objects_detected')}")
        print(f"  alerts: {ping.get('alerts')}")
    
    print("\n" + "="*70)
    print("✅ TEST COMPLETE - All logic working!")
    print("   You can now run the main generator to save files.")
    print("="*70)

if __name__ == "__main__":
    main()


🧪 TESTING GENERATOR - No Files Will Be Saved

📦 Generating 100 traditional trucks (20 per OEM)...
   ✅ Generated 100 traditional trucks
📦 Generating 20 autonomous pods...
   ✅ Generated 20 autonomous pods

📡 Generating 100 traditional telemetry rows...
   ✅ Generated 100 rows

📡 Generating 100 autonomous telemetry rows...
   ✅ Generated 100 rows

----------------------------------------------------------------------
📋 SAMPLE TRADITIONAL TELEMETRY (First 2 rows):
----------------------------------------------------------------------

Row 1 (OEM: DAF):
  event_id: 6b425f40-504b-4906-b34e-edc790e3123a
  truck_id: EIN-TRK-DAF-006
  timestamp: 2026-08-24T19:14:11Z
  latitude: 57.979371
  longitude: 12.532849
  alerts: []

Row 2 (OEM: BYD):
  event_id: 9bd61c92-9eab-40aa-8128-d2c500af9004
  truck_id: EIN-TRK-BYD-009
  timestamp: 2026-08-24T08:58:32Z
  latitude: 56.01001
  longitude: 13.219044
  alerts: []

----------------------------------------------------------------------
📋 SAMPLE AUTON

OK generating the historical data now


In [3]:
"""
generate_github_sources.py

MAIN GENERATOR - Saves files to github_sources/ folder.
Traditional: 100,000 rows | Autonomous: 100,000 rows
"""

import json
import random
import uuid
import os
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('sv_SE')

# ===================================================================
# CONFIGURATION - MAIN GENERATION
# ===================================================================

NUM_TRUCKS_PER_OEM = 20
NUM_OEMS = 5
NUM_AUTONOMOUS = 20

ROWS_TRADITIONAL = 100000
ROWS_AUTONOMOUS = 100000

# ===================================================================
# OEM DISTRIBUTION (Same as test)
# ===================================================================

OEMS = {
    "Tesla": {
        "models": ["Semi Electric"],
        "battery_options": [500, 600, 750],
        "telemetry_fields": {
            "energy": "battery_energy_used",
            "energy_type": "cumulative",
            "soc": "soc_percentage",
            "speed": "vehicle_speed",
            "location": "gps_coordinates"
        }
    },
    "BYD": {
        "models": ["8TT", "8R"],
        "battery_options": [300, 350, 400],
        "telemetry_fields": {
            "energy": "energy_consumption",
            "energy_type": "cumulative",
            "soc": "battery_soc",
            "speed": "speed_kmh",
            "location": "location"
        }
    },
    "Scania": {
        "models": ["BEV 40R", "BEV 45R"],
        "battery_options": [300, 450, 600],
        "telemetry_fields": {
            "energy": "Total Discharged Energy",
            "energy_type": "cumulative",
            "soc": "EV Battery State Of Charge",
            "speed": "Vehicle Speed",
            "location": "coordinates"
        }
    },
    "Mercedes": {
        "models": ["eActros 300", "eActros 400"],
        "battery_options": [300, 400],
        "telemetry_fields": {
            "energy": "energy_consumed",
            "energy_type": "cumulative",
            "soc": "soc",
            "speed": "speed",
            "location": "gps"
        }
    },
    "DAF": {
        "models": ["CF Electric", "XF Electric"],
        "battery_options": [300, 450],
        "telemetry_fields": {
            "energy": "Energy consumed",
            "energy_type": "fms_standard",
            "soc": "State of charge",
            "speed": "Vehicle speed",
            "location": "position"
        }
    }
}

# ===================================================================
# GENERATE TRUCKS (Traditional)
# ===================================================================

def generate_traditional_trucks():
    """Generates 100 trucks across 5 OEMs."""
    trucks = []
    for oem_name, oem_data in OEMS.items():
        for i in range(NUM_TRUCKS_PER_OEM):
            purchase_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 1000))
            age_days = (datetime(2026, 8, 25) - purchase_date).days
            soh = max(85, min(100, 100 - age_days * 0.005))
            
            trucks.append({
                "truck_id": f"EIN-TRK-{oem_name[:3].upper()}-{i+1:03d}",
                "vin": fake.vin(),
                "oem": oem_name,
                "model": random.choice(oem_data["models"]),
                "battery_capacity_kwh": random.choice(oem_data["battery_options"]),
                "purchase_date": purchase_date.isoformat(),
                "home_depot_id": random.choice(["STO", "GOT", "MAL", "JKP", "HEL", "ORE"]),
                "vehicle_type": "human_driven",
                "initial_state_of_health_pct": round(soh, 1)
            })
    return trucks

# ===================================================================
# GENERATE AUTONOMOUS PODS
# ===================================================================

def generate_autonomous_pods():
    """Generates 20 autonomous pods."""
    pods = []
    for i in range(NUM_AUTONOMOUS):
        purchase_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 600))
        age_days = (datetime(2026, 8, 25) - purchase_date).days
        soh = max(92, min(100, 100 - age_days * 0.003))
        
        pods.append({
            "pod_id": f"EIN-POD-{i+1:03d}",
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "battery_capacity_kwh": random.choice([250, 300, 320]),
            "sensor_suite": ["lidar", "radar", "cameras", "gps", "inertial"],
            "autonomy_level": "Level 4",
            "max_range_km": random.choice([500, 600, 650]),
            "deployment_date": purchase_date.isoformat(),
            "home_depot_id": random.choice(["STO", "GOT", "MAL", "JKP"]),
            "initial_state_of_health_pct": round(soh, 1)
        })
    return pods

# ===================================================================
# GENERATE TRADITIONAL TELEMETRY (Full)
# ===================================================================

def generate_traditional_telemetry(trucks, num_rows=ROWS_TRADITIONAL):
    """Generates full traditional telemetry."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        truck = random.choice(trucks)
        oem = truck["oem"]
        oem_fields = OEMS[oem]["telemetry_fields"]
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "truck_id": truck["truck_id"],
            "oem": oem,
            "vehicle_type": "human_driven",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            oem_fields["soc"]: round(random.uniform(20, 95), 1),
            oem_fields["speed"]: round(random.uniform(30, 85), 1),
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "ambient_temp_c": round(random.uniform(-5, 20), 1),
            "cargo_weight_kg": random.randint(5000, 30000),
            "is_charging": random.random() < 0.1,
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # OEM-specific energy field
        payload[oem_fields["energy"]] = round(random.uniform(0, 5000), 2)

        # Planted defects
        if random.random() < 0.0005:
            payload[oem_fields["soc"]] = random.choice([0, 100])
        if random.random() < 0.008:
            payload["cargo_weight_kg"] = None
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload[oem_fields["soc"]] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

        if (i + 1) % 10000 == 0:
            print(f"  Traditional: Generated {i+1:,} rows...")

    return pings

# ===================================================================
# GENERATE AUTONOMOUS TELEMETRY (Full)
# ===================================================================

def generate_autonomous_telemetry(pods, num_rows=ROWS_AUTONOMOUS):
    """Generates full autonomous pod telemetry."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        pod = random.choice(pods)
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "pod_id": pod["pod_id"],
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "speed_kmh": round(random.uniform(30, 85), 1),
            "state_of_charge_pct": round(random.uniform(20, 95), 1),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "is_charging": random.random() < 0.1,
            "cargo_weight_kg": random.randint(5000, 30000),
            
            # Autonomous-specific fields
            "perception": {
                "objects_detected": random.randint(0, 25),
                "closest_object_distance_m": round(random.uniform(5, 150), 1)
            },
            "path_planning": {
                "planned_steering_angle_deg": round(random.uniform(-10, 10), 1),
                "planned_acceleration_ms2": round(random.uniform(-2, 2), 1)
            },
            "safety": {
                "system_health": "normal",
                "fallback_mode_active": False,
                "emergency_stop_triggered": False
            },
            "mission_status": random.choice(["en_route", "charging", "awaiting_remote", "maintenance"]),
            "remote_monitor_id": f"RM-{random.randint(1, 50):04d}",
            "safety_confidence_score": round(random.uniform(0.85, 0.99), 2),
            
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # Planted defects
        if random.random() < 0.0005:
            payload["state_of_charge_pct"] = random.choice([0, 100])
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload["state_of_charge_pct"] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

        if (i + 1) % 10000 == 0:
            print(f"  Autonomous: Generated {i+1:,} rows...")

    return pings

# ===================================================================
# WRITE FUNCTIONS
# ===================================================================

def write_json(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"✅ Wrote {filepath}")

def write_ndjson(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"✅ Wrote {len(data):,} rows to {filepath} ({size_mb:.2f} MB)")

# ===================================================================
# MAIN
# ===================================================================

def main():
    print("\n" + "="*70)
    print("🚀 GENERATING GITHUB SOURCE FILES")
    print(f"   Traditional: {ROWS_TRADITIONAL:,} rows")
    print(f"   Autonomous: {ROWS_AUTONOMOUS:,} rows")
    print("="*70 + "\n")

    # 1. Generate mapping data
    print("📦 Generating trucks and pods...")
    trucks = generate_traditional_trucks()
    pods = generate_autonomous_pods()
    print(f"   ✅ {len(trucks)} traditional trucks (20 per OEM)")
    print(f"   ✅ {len(pods)} autonomous pods")

    # 2. Write mapping files
    print("\n📝 Writing mapping files...")
    write_json(trucks, "github_sources/mapping/trucks.json")
    write_json(pods, "github_sources/mapping/autonomous_pods.json")
    write_json(OEMS, "github_sources/mapping/oem_config.json")

    # 3. Generate telemetry
    print(f"\n📡 Generating {ROWS_TRADITIONAL:,} traditional telemetry rows...")
    print("   (This may take a few minutes)")
    traditional_pings = generate_traditional_telemetry(trucks, ROWS_TRADITIONAL)
    
    print(f"\n📡 Generating {ROWS_AUTONOMOUS:,} autonomous telemetry rows...")
    print("   (This may take a few minutes)")
    autonomous_pings = generate_autonomous_telemetry(pods, ROWS_AUTONOMOUS)

    # 4. Write telemetry
    print("\n💾 Saving files...")
    write_ndjson(traditional_pings, "github_sources/historical/traditional_sample.ndjson")
    write_ndjson(autonomous_pings, "github_sources/historical/autonomous_sample.ndjson")

    print("\n" + "="*70)
    print("🎉 DONE! Upload 'github_sources/' folder to GitHub.")
    print("="*70)
    print("\n📁 Files generated:")
    print("   github_sources/mapping/trucks.json")
    print("   github_sources/mapping/autonomous_pods.json")
    print("   github_sources/mapping/oem_config.json")
    print("   github_sources/historical/traditional_sample.ndjson")
    print("   github_sources/historical/autonomous_sample.ndjson")

if __name__ == "__main__":
    main()


🚀 GENERATING GITHUB SOURCE FILES
   Traditional: 100,000 rows
   Autonomous: 100,000 rows

📦 Generating trucks and pods...
   ✅ 100 traditional trucks (20 per OEM)
   ✅ 20 autonomous pods

📝 Writing mapping files...
✅ Wrote github_sources/mapping/trucks.json
✅ Wrote github_sources/mapping/autonomous_pods.json
✅ Wrote github_sources/mapping/oem_config.json

📡 Generating 100,000 traditional telemetry rows...
   (This may take a few minutes)
  Traditional: Generated 10,000 rows...
  Traditional: Generated 20,000 rows...
  Traditional: Generated 30,000 rows...
  Traditional: Generated 40,000 rows...
  Traditional: Generated 50,000 rows...
  Traditional: Generated 60,000 rows...
  Traditional: Generated 70,000 rows...
  Traditional: Generated 80,000 rows...
  Traditional: Generated 90,000 rows...
  Traditional: Generated 100,000 rows...

📡 Generating 100,000 autonomous telemetry rows...
   (This may take a few minutes)
  Autonomous: Generated 10,000 rows...
  Autonomous: Generated 20,000 r

In [4]:
"""
regenerate_sample.py

REGENERATE ONLY THE TELEMETRY SAMPLE FILES
Uses existing mapping files from github_sources/mapping/
Generates 50,000 rows each for traditional and autonomous.

Run this if you already generated the mapping files but need smaller sample files.
"""

import json
import random
import uuid
import os
from datetime import datetime, timedelta

# ===================================================================
# CONFIGURATION
# ===================================================================

ROWS_TRADITIONAL = 50000
ROWS_AUTONOMOUS = 50000

# ===================================================================
# LOAD EXISTING MAPPING FILES
# ===================================================================

def load_mapping_files():
    """Load the mapping files you already generated."""
    
    # Load trucks
    with open("github_sources/mapping/trucks.json", "r") as f:
        trucks = json.load(f)
    
    # Load autonomous pods
    with open("github_sources/mapping/autonomous_pods.json", "r") as f:
        pods = json.load(f)
    
    # Load OEM config
    with open("github_sources/mapping/oem_config.json", "r") as f:
        oem_config = json.load(f)
    
    print(f"✅ Loaded {len(trucks)} traditional trucks")
    print(f"✅ Loaded {len(pods)} autonomous pods")
    print(f"✅ Loaded {len(oem_config)} OEM configs")
    
    return trucks, pods, oem_config

# ===================================================================
# GENERATE TRADITIONAL TELEMETRY
# ===================================================================

def generate_traditional_telemetry(trucks, oem_config, num_rows=ROWS_TRADITIONAL):
    """Generates traditional telemetry using existing trucks."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        truck = random.choice(trucks)
        oem = truck["oem"]
        oem_fields = oem_config[oem]["telemetry_fields"]
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "truck_id": truck["truck_id"],
            "oem": oem,
            "vehicle_type": "human_driven",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            oem_fields["soc"]: round(random.uniform(20, 95), 1),
            oem_fields["speed"]: round(random.uniform(30, 85), 1),
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "ambient_temp_c": round(random.uniform(-5, 20), 1),
            "cargo_weight_kg": random.randint(5000, 30000),
            "is_charging": random.random() < 0.1,
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # OEM-specific energy field
        payload[oem_fields["energy"]] = round(random.uniform(0, 5000), 2)

        # Planted defects
        if random.random() < 0.0005:
            payload[oem_fields["soc"]] = random.choice([0, 100])
        if random.random() < 0.008:
            payload["cargo_weight_kg"] = None
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload[oem_fields["soc"]] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

        if (i + 1) % 10000 == 0:
            print(f"  Traditional: Generated {i+1:,} rows...")

    return pings

# ===================================================================
# GENERATE AUTONOMOUS TELEMETRY
# ===================================================================

def generate_autonomous_telemetry(pods, num_rows=ROWS_AUTONOMOUS):
    """Generates autonomous telemetry using existing pods."""
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        pod = random.choice(pods)
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "pod_id": pod["pod_id"],
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "speed_kmh": round(random.uniform(30, 85), 1),
            "state_of_charge_pct": round(random.uniform(20, 95), 1),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "is_charging": random.random() < 0.1,
            "cargo_weight_kg": random.randint(5000, 30000),
            
            "perception": {
                "objects_detected": random.randint(0, 25),
                "closest_object_distance_m": round(random.uniform(5, 150), 1)
            },
            "path_planning": {
                "planned_steering_angle_deg": round(random.uniform(-10, 10), 1),
                "planned_acceleration_ms2": round(random.uniform(-2, 2), 1)
            },
            "safety": {
                "system_health": "normal",
                "fallback_mode_active": False,
                "emergency_stop_triggered": False
            },
            "mission_status": random.choice(["en_route", "charging", "awaiting_remote", "maintenance"]),
            "remote_monitor_id": f"RM-{random.randint(1, 50):04d}",
            "safety_confidence_score": round(random.uniform(0.85, 0.99), 2),
            
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # Planted defects
        if random.random() < 0.0005:
            payload["state_of_charge_pct"] = random.choice([0, 100])
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload["state_of_charge_pct"] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

        if (i + 1) % 10000 == 0:
            print(f"  Autonomous: Generated {i+1:,} rows...")

    return pings

# ===================================================================
# WRITE NDJSON
# ===================================================================

def write_ndjson(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"✅ Wrote {len(data):,} rows to {filepath} ({size_mb:.2f} MB)")

# ===================================================================
# MAIN
# ===================================================================

def main():
    print("\n" + "="*70)
    print("🔄 REGENERATING TELEMETRY SAMPLE FILES ONLY")
    print("   Using existing mapping files")
    print(f"   Traditional: {ROWS_TRADITIONAL:,} rows")
    print(f"   Autonomous: {ROWS_AUTONOMOUS:,} rows")
    print("="*70 + "\n")

    # 1. Load existing mapping files
    print("📂 Loading existing mapping files...")
    trucks, pods, oem_config = load_mapping_files()

    # 2. Generate traditional telemetry
    print(f"\n📡 Generating {ROWS_TRADITIONAL:,} traditional telemetry rows...")
    print("   (This may take 1-2 minutes)")
    traditional_pings = generate_traditional_telemetry(trucks, oem_config, ROWS_TRADITIONAL)

    # 3. Generate autonomous telemetry
    print(f"\n📡 Generating {ROWS_AUTONOMOUS:,} autonomous telemetry rows...")
    print("   (This may take 1-2 minutes)")
    autonomous_pings = generate_autonomous_telemetry(pods, ROWS_AUTONOMOUS)

    # 4. Write the files (overwrites existing)
    print("\n💾 Saving files (overwriting existing)...")
    write_ndjson(traditional_pings, "github_sources/historical/traditional_sample.ndjson")
    write_ndjson(autonomous_pings, "github_sources/historical/autonomous_sample.ndjson")

    print("\n" + "="*70)
    print("✅ DONE! Sample files regenerated with smaller sizes.")
    print("   Files are now safe to upload to GitHub.")
    print("="*70)

if __name__ == "__main__":
    main()


🔄 REGENERATING TELEMETRY SAMPLE FILES ONLY
   Using existing mapping files
   Traditional: 50,000 rows
   Autonomous: 50,000 rows

📂 Loading existing mapping files...
✅ Loaded 100 traditional trucks
✅ Loaded 20 autonomous pods
✅ Loaded 5 OEM configs

📡 Generating 50,000 traditional telemetry rows...
   (This may take 1-2 minutes)
  Traditional: Generated 10,000 rows...
  Traditional: Generated 20,000 rows...
  Traditional: Generated 30,000 rows...
  Traditional: Generated 40,000 rows...
  Traditional: Generated 50,000 rows...

📡 Generating 50,000 autonomous telemetry rows...
   (This may take 1-2 minutes)
  Autonomous: Generated 10,000 rows...
  Autonomous: Generated 20,000 rows...
  Autonomous: Generated 30,000 rows...
  Autonomous: Generated 40,000 rows...
  Autonomous: Generated 50,000 rows...

💾 Saving files (overwriting existing)...
✅ Wrote 50,000 rows to github_sources/historical/traditional_sample.ndjson (23.64 MB)
✅ Wrote 50,000 rows to github_sources/historical/autonomous_sam

In [5]:
"""
regenerate_autonomous_only.py

REGENERATES ONLY THE AUTONOMOUS SAMPLE FILE
Generates 25,000 rows (~22 MB) - safe for GitHub.
"""

import json
import random
import uuid
import os
from datetime import datetime, timedelta

# ===================================================================
# CONFIGURATION
# ===================================================================

ROWS_AUTONOMOUS = 25000  # Target: ~22 MB

# ===================================================================
# LOAD AUTONOMOUS PODS
# ===================================================================

def load_autonomous_pods():
    with open("github_sources/mapping/autonomous_pods.json", "r") as f:
        pods = json.load(f)
    print(f"✅ Loaded {len(pods)} autonomous pods")
    return pods

# ===================================================================
# GENERATE AUTONOMOUS TELEMETRY
# ===================================================================

def generate_autonomous_telemetry(pods, num_rows=ROWS_AUTONOMOUS):
    waypoints = [
        (59.3293, 18.0686), (57.7089, 11.9746), (55.6050, 13.0038),
        (57.7828, 14.1612), (56.0467, 12.6944), (59.2748, 15.2066)
    ]
    pings = []
    start_date = datetime(2026, 8, 24, 6, 0, 0)

    for i in range(num_rows):
        pod = random.choice(pods)
        route_start = random.choice(waypoints)
        route_end = random.choice(waypoints)
        offset = random.randint(0, 16 * 3600)
        timestamp = start_date + timedelta(seconds=offset)
        progress = offset / (16 * 3600)

        payload = {
            "event_id": str(uuid.uuid4()),
            "pod_id": pod["pod_id"],
            "oem": "Einride",
            "vehicle_type": "autonomous_pod",
            "timestamp": timestamp.isoformat() + "Z",
            "ingested_by": "batch_archive",
            
            "latitude": round(route_start[0] + (route_end[0] - route_start[0]) * progress, 6),
            "longitude": round(route_start[1] + (route_end[1] - route_start[1]) * progress, 6),
            "speed_kmh": round(random.uniform(30, 85), 1),
            "state_of_charge_pct": round(random.uniform(20, 95), 1),
            "battery_temp_c": round(20 + random.uniform(-5, 15), 1),
            "cumulative_energy_kwh": round(random.uniform(0, 5000), 2),
            "is_charging": random.random() < 0.1,
            "cargo_weight_kg": random.randint(5000, 30000),
            
            "perception": {
                "objects_detected": random.randint(0, 25),
                "closest_object_distance_m": round(random.uniform(5, 150), 1)
            },
            "path_planning": {
                "planned_steering_angle_deg": round(random.uniform(-10, 10), 1),
                "planned_acceleration_ms2": round(random.uniform(-2, 2), 1)
            },
            "safety": {
                "system_health": "normal",
                "fallback_mode_active": False,
                "emergency_stop_triggered": False
            },
            "mission_status": random.choice(["en_route", "charging", "awaiting_remote", "maintenance"]),
            "remote_monitor_id": f"RM-{random.randint(1, 50):04d}",
            "safety_confidence_score": round(random.uniform(0.85, 0.99), 2),
            
            "weather_condition": random.choice(["clear", "cloudy", "rain", "snow"]),
            "alerts": []
        }

        # Planted defects
        if random.random() < 0.0005:
            payload["state_of_charge_pct"] = random.choice([0, 100])
        if random.random() < 0.001:
            payload["latitude"] = 0.0
            payload["longitude"] = 0.0

        if payload["state_of_charge_pct"] < 20:
            payload["alerts"].append({"code": "WARN_LOW_SOC", "severity": "medium"})
        if payload["battery_temp_c"] < 5:
            payload["alerts"].append({"code": "WARN_BATTERY_COLD", "severity": "low"})

        pings.append(payload)

        if (i + 1) % 5000 == 0:
            print(f"  Generated {i+1:,} rows...")

    return pings

# ===================================================================
# WRITE NDJSON
# ===================================================================

def write_ndjson(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"✅ Wrote {len(data):,} rows to {filepath} ({size_mb:.2f} MB)")

# ===================================================================
# MAIN
# ===================================================================

def main():
    print("\n" + "="*70)
    print("🔄 REGENERATING AUTONOMOUS SAMPLE ONLY")
    print(f"   Autonomous: {ROWS_AUTONOMOUS:,} rows")
    print("   Target size: ~22 MB")
    print("="*70 + "\n")

    pods = load_autonomous_pods()

    print(f"\n📡 Generating {ROWS_AUTONOMOUS:,} autonomous telemetry rows...")
    autonomous_pings = generate_autonomous_telemetry(pods, ROWS_AUTONOMOUS)

    print("\n💾 Saving file...")
    write_ndjson(autonomous_pings, "github_sources/historical/autonomous_sample.ndjson")

    print("\n" + "="*70)
    print("✅ DONE! File size should be ~22 MB.")
    print(f"   {ROWS_AUTONOMOUS:,} rows. Safe for GitHub.")
    print("="*70)

if __name__ == "__main__":
    main()


🔄 REGENERATING AUTONOMOUS SAMPLE ONLY
   Autonomous: 25,000 rows
   Target size: ~22 MB

✅ Loaded 20 autonomous pods

📡 Generating 25,000 autonomous telemetry rows...
  Generated 5,000 rows...
  Generated 10,000 rows...
  Generated 15,000 rows...
  Generated 20,000 rows...
  Generated 25,000 rows...

💾 Saving file...
✅ Wrote 25,000 rows to github_sources/historical/autonomous_sample.ndjson (19.28 MB)

✅ DONE! File size should be ~22 MB.
   25,000 rows. Safe for GitHub.


In [ ]:
"""
generate_missing_mappings.py

Generates all missing mapping files:
- customers.json
- routes.json
- depots.json
- drivers.json
- tariffs.json
- grid_carbon_intensity.json
- battery_specs.json
- charger_types.json
- vehicle_types.json
- weather_mapping.json
- sla_tiers.json

Run this after generating the main files.
"""

import json
import random
import os
from datetime import datetime, timedelta
from faker import Faker

fake = Faker('sv_SE')

# ===================================================================
# 1. CUSTOMERS
# ===================================================================

def generate_customers():
    customers = [
        {"name": "Oatly Sverige", "industry": "Food", "sla": 98.5, "sustainability": True, "baseline": 85},
        {"name": "ICA Sverige", "industry": "Retail", "sla": 99.0, "sustainability": True, "baseline": 80},
        {"name": "Electrolux", "industry": "Manufacturing", "sla": 97.0, "sustainability": False, "baseline": 90},
        {"name": "SCA", "industry": "Paper", "sla": 95.5, "sustainability": True, "baseline": 95},
        {"name": "IKEA Sverige", "industry": "Retail", "sla": 99.5, "sustainability": True, "baseline": 75},
        {"name": "Volvo Cars", "industry": "Automotive", "sla": 98.0, "sustainability": False, "baseline": 88},
        {"name": "Skånemejerier", "industry": "Food", "sla": 97.5, "sustainability": True, "baseline": 82},
        {"name": "Stora Enso", "industry": "Paper", "sla": 96.0, "sustainability": True, "baseline": 92},
        {"name": "H&M", "industry": "Retail", "sla": 98.0, "sustainability": True, "baseline": 78},
        {"name": "Ahlgrens", "industry": "Food", "sla": 96.5, "sustainability": False, "baseline": 88},
        {"name": "Saab", "industry": "Manufacturing", "sla": 99.0, "sustainability": False, "baseline": 85},
        {"name": "Tetra Pak", "industry": "Manufacturing", "sla": 97.0, "sustainability": True, "baseline": 80},
        {"name": "Systembolaget", "industry": "Retail", "sla": 99.5, "sustainability": True, "baseline": 75},
        {"name": "LKAB", "industry": "Mining", "sla": 95.0, "sustainability": True, "baseline": 100},
        {"name": "Biltema", "industry": "Retail", "sla": 97.0, "sustainability": False, "baseline": 82},
    ]
    return [{
        "customer_id": f"CUST-{i+1:02d}",
        "customer_name": c["name"],
        "industry": c["industry"],
        "contract_start_date": "2024-01-01",
        "sla_otif_target_pct": c["sla"],
        "sustainability_reporting_required": c["sustainability"],
        "diesel_baseline_gco2_per_tonkm": c["baseline"]
    } for i, c in enumerate(customers)]

# ===================================================================
# 2. ROUTES
# ===================================================================

def generate_routes():
    return [
        {"route_id": "SE-E4-STO-JKP", "name": "Stockholm → Jönköping (E4)", "origin": "STO", "dest": "JKP", "distance_km": 320, "elevation_gain_m": 105},
        {"route_id": "SE-E4-JKP-GOT", "name": "Jönköping → Gothenburg (E4/40)", "origin": "JKP", "dest": "GOT", "distance_km": 150, "elevation_gain_m": 90},
        {"route_id": "SE-E6-GOT-MAL", "name": "Gothenburg → Malmö (E6)", "origin": "GOT", "dest": "MAL", "distance_km": 270, "elevation_gain_m": 90},
        {"route_id": "SE-E6-MAL-HEL", "name": "Malmö → Helsingborg (E6)", "origin": "MAL", "dest": "HEL", "distance_km": 65, "elevation_gain_m": 50},
        {"route_id": "SE-E20-STO-ORE", "name": "Stockholm → Örebro (E18/E20)", "origin": "STO", "dest": "ORE", "distance_km": 200, "elevation_gain_m": 65},
        {"route_id": "SE-URB-GOT", "name": "Gothenburg urban distribution loop", "origin": "GOT", "dest": "GOT", "distance_km": 45, "elevation_gain_m": 15},
    ]

# ===================================================================
# 3. DEPOTS
# ===================================================================

def generate_depots():
    depots = {
        "STO": ("Stockholm", 59.3293, 18.0686, "SE1", 12, 200),
        "GOT": ("Gothenburg", 57.7089, 11.9746, "SE3", 10, 150),
        "MAL": ("Malmö", 55.6050, 13.0038, "SE4", 8, 150),
        "JKP": ("Jönköping", 57.7828, 14.1612, "SE3", 6, 120),
        "HEL": ("Helsingborg", 56.0467, 12.6944, "SE4", 4, 120),
        "ORE": ("Örebro", 59.2748, 15.2066, "SE2", 4, 120),
    }
    return [{
        "depot_id": dep_id,
        "depot_name": name,
        "latitude": lat,
        "longitude": lon,
        "region": zone,
        "charger_count": chargers,
        "max_charger_power_kw": power
    } for dep_id, (name, lat, lon, zone, chargers, power) in depots.items()]

# ===================================================================
# 4. DRIVERS
# ===================================================================

def generate_drivers():
    return [{
        "driver_id": f"DRV-{i+1:04d}",
        "driver_name": fake.name(),
        "home_depot_id": random.choice(["STO", "GOT", "MAL", "JKP", "HEL", "ORE"]),
        "license_class": random.choice(["C", "CE"]),
        "hire_date": (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 600))).isoformat()
    } for i in range(150)]

# ===================================================================
# 5. TARIFFS
# ===================================================================

def generate_tariffs():
    zones = ["SE1", "SE2", "SE3", "SE4"]
    day_types = ["weekday", "weekend"]
    data = []
    for zone in zones:
        base = random.uniform(0.8, 1.2)
        for day_type in day_types:
            for hour in range(24):
                if day_type == "weekday" and 8 <= hour < 20:
                    price = base * random.uniform(1.5, 2.5)
                elif day_type == "weekday":
                    price = base * random.uniform(0.6, 1.0)
                else:
                    price = base * random.uniform(0.8, 1.2)
                data.append({
                    "price_zone": zone,
                    "hour_of_day": hour,
                    "day_type": day_type,
                    "price_sek_per_kwh": round(price, 4)
                })
    return data

# ===================================================================
# 6. GRID CARBON INTENSITY
# ===================================================================

def generate_grid_carbon():
    zones = ["SE1", "SE2", "SE3", "SE4"]
    data = []
    start = datetime(2026, 8, 24, 0, 0, 0)
    for zone in zones:
        base = random.uniform(30, 80)
        for hour in range(24):
            dt = start + timedelta(hours=hour)
            solar = 1 - 0.3 * max(0, 1 - abs(hour - 13) / 5) if 8 <= hour <= 18 else 1
            intensity = base * solar * random.uniform(0.7, 1.3) * random.uniform(0.9, 1.1)
            data.append({
                "price_zone": zone,
                "datetime_hour": dt.isoformat(),
                "carbon_intensity_gco2_per_kwh": round(max(10, min(150, intensity)), 1)
            })
    return data

# ===================================================================
# 7. BATTERY SPECS
# ===================================================================

def generate_battery_specs():
    return {
        "battery_chemistries": {
            "NMC": {"degradation_rate": 0.0002, "cycle_life": 1500, "max_charge_rate_c": 1.5},
            "LFP": {"degradation_rate": 0.00015, "cycle_life": 3000, "max_charge_rate_c": 1.0},
            "NCA": {"degradation_rate": 0.00025, "cycle_life": 1000, "max_charge_rate_c": 2.0}
        }
    }

# ===================================================================
# 8. CHARGER TYPES
# ===================================================================

def generate_charger_types():
    return {
        "charger_types": {
            "Level_2_AC": {"power_kw": 22, "connector": "Type2", "typical_duration_hours": 8},
            "DC_Fast_50": {"power_kw": 50, "connector": "CCS2", "typical_duration_hours": 1.5},
            "DC_Fast_150": {"power_kw": 150, "connector": "CCS2", "typical_duration_hours": 0.8},
            "DC_Ultra_350": {"power_kw": 350, "connector": "CCS2", "typical_duration_hours": 0.4}
        }
    }

# ===================================================================
# 9. VEHICLE TYPES
# ===================================================================

def generate_vehicle_types():
    return {
        "vehicle_types": {
            "human_driven": {
                "data_sources": ["telematics", "driver_app", "tms"],
                "ping_interval_seconds": 60,
                "has_driver": True
            },
            "autonomous_pod": {
                "data_sources": ["telematics", "lidar", "radar", "cameras", "gps", "inertial"],
                "ping_interval_seconds": 5,
                "has_driver": False
            }
        }
    }

# ===================================================================
# 10. WEATHER MAPPING
# ===================================================================

def generate_weather_mapping():
    return {
        "weather_conditions": {
            "clear": {"consumption_factor": 1.0, "regen_efficiency": 0.6},
            "cloudy": {"consumption_factor": 1.02, "regen_efficiency": 0.6},
            "rain": {"consumption_factor": 1.08, "regen_efficiency": 0.55, "rolling_resistance": 1.05},
            "snow": {"consumption_factor": 1.15, "regen_efficiency": 0.5, "rolling_resistance": 1.15}
        }
    }

# ===================================================================
# 11. SLA TIERS
# ===================================================================

def generate_sla_tiers():
    return {
        "sla_tiers": {
            "premium": {"otif_target": 99.5, "penalty_rate": 0.05},
            "standard": {"otif_target": 98.0, "penalty_rate": 0.02},
            "basic": {"otif_target": 95.0, "penalty_rate": 0.01}
        }
    }

# ===================================================================
# WRITE FUNCTION
# ===================================================================

def write_json(data, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"✅ Wrote {os.path.basename(filepath)} ({size_kb:.1f} KB)")

# ===================================================================
# MAIN
# ===================================================================

def main():
    print("\n" + "="*70)
    print("📝 GENERATING MISSING MAPPING FILES")
    print("="*70 + "\n")

    files = [
        ("customers.json", generate_customers()),
        ("routes.json", generate_routes()),
        ("depots.json", generate_depots()),
        ("drivers.json", generate_drivers()),
        ("tariffs.json", generate_tariffs()),
        ("grid_carbon_intensity.json", generate_grid_carbon()),
        ("battery_specs.json", generate_battery_specs()),
        ("charger_types.json", generate_charger_types()),
        ("vehicle_types.json", generate_vehicle_types()),
        ("weather_mapping.json", generate_weather_mapping()),
        ("sla_tiers.json", generate_sla_tiers()),
    ]

    for filename, data in files:
        write_json(data, f"github_sources/mapping/{filename}")

    print("\n" + "="*70)
    print("✅ DONE! All mapping files generated.")
    print("📁 github_sources/mapping/ now contains 14 files:")
    print("   - oem_config.json")
    print("   - trucks.json")
    print("   - autonomous_pods.json")
    print("   - customers.json")
    print("   - routes.json")
    print("   - depots.json")
    print("   - drivers.json")
    print("   - tariffs.json")
    print("   - grid_carbon_intensity.json")
    print("   - battery_specs.json")
    print("   - charger_types.json")
    print("   - vehicle_types.json")
    print("   - weather_mapping.json")
    print("   - sla_tiers.json")
    print("="*70)

if __name__ == "__main__":
    main()